# Facial Image Segmentation with U-Net

**Bita Moharrami · University of Tehran · Original project: August 2025**

Train and evaluate the original U-Net architecture on EasyPortrait. See the repository README for installation, recorded results and reproducibility notes. This cleaned workflow has no saved execution outputs; historical evidence is preserved separately in `docs/`, `results/` and `assets/`. Running all cells starts a new training run.

In [ ]:
from pathlib import Path
import os
import random
import kagglehub
import tensorflow as tf
from tensorflow.keras import layers, Model
import numpy as np
import matplotlib.pyplot as plt
import cv2

# Works when Jupyter starts in either the repository root or notebooks/.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
MODEL_DIR = ROOT / "models"
OUTPUT_DIR = ROOT / "outputs"
MODEL_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

# Explicit defaults for new runs; historical results were not rerun in cleanup.
SEED = 42
EPOCHS = 18
USE_AUGMENTATION = False
tf.keras.utils.set_random_seed(SEED)

In [ ]:
IMG_HEIGHT = 256
IMG_WIDTH = 256
BATCH_SIZE = 16
NUM_CLASSES = 9


COLORS = [
    (0, 128, 128),    # 0 Background
    (128, 0, 0),      # 1 Person
    (255, 224, 189),  # 2 Skin
    (0, 255, 0),      # 3 Left Brow
    (0, 128, 0),      # 4 Right Brow
    (0, 0, 255),      # 5 Left Eye
    (0, 0, 128),      # 6 Right Eye
    (255, 0, 0),      # 7 Lips
    (255, 255, 255),  # 8 Teeth
]

## Dataset and preprocessing

Masks use integer class IDs 0–8. Files are paired by stem; non-image metadata files are ignored.

In [ ]:
# Set EASYPORTRAIT_DIR to reuse a local dataset, or let kagglehub download it.
dataset_path = Path(os.environ["EASYPORTRAIT_DIR"]).expanduser() if os.environ.get("EASYPORTRAIT_DIR") else Path(kagglehub.dataset_download("kapitanov/easyportrait"))
for folder in ("images", "annotations"):
    if not (dataset_path / folder).is_dir():
        raise FileNotFoundError(f"Expected {folder}/ under {dataset_path}")
print("Dataset:", dataset_path)

def paired_paths(images_dir, masks_dir):
    images = sorted(p for p in Path(images_dir).iterdir() if p.suffix.lower() in {".jpg", ".jpeg"})
    masks = {p.stem: p for p in Path(masks_dir).iterdir() if p.suffix.lower() == ".png"}
    if not images:
        raise ValueError(f"No JPEG images in {images_dir}")
    if len({p.stem for p in images}) != len(images):
        raise ValueError("Duplicate image stems would create ambiguous mask pairs")
    missing = [p.name for p in images if p.stem not in masks]
    if missing:
        raise ValueError(f"Missing masks for: {missing[:5]}")
    return [str(p) for p in images], [str(masks[p.stem]) for p in images]

for split in ("train", "val", "test"):
    images, masks = paired_paths(dataset_path / "images" / split, dataset_path / "annotations" / split)
    print(f"{split}: {len(images)} matched image/mask pairs")

In [ ]:
def process_image(img_path, mask_path):
    """
    Reads and preprocesses a single image-mask pair for segmentation.

    Args :
    img_path : Path to the image file (JPEG)
    mask_path : Path to the mask file (PNG)

    Returns :
    img : Preprocessed image tensor of shape (H, W, 3), float32 in [0,1]
    mask : Preprocessed mask tensor of shape (H, W, 1), uint8 with class labels
    """


    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)  # Decode JPEG into RGB tensor
    img = tf.image.resize(img, [IMG_HEIGHT, IMG_WIDTH])  # Resize to fixed input size
    img = tf.cast(img, tf.float32) / 255.0  # Normalize pixel values [0,1]



    mask = tf.io.read_file(mask_path)
    mask = tf.image.decode_png(mask, channels=1)  # Decode PNG (single channel labels)
    mask = tf.image.resize(mask, [IMG_HEIGHT, IMG_WIDTH], method="nearest")  # Resize mask to input size
    mask = tf.cast(mask, tf.uint8)  # Ensure mask is integer labels


    return img, mask

In [ ]:
def augment(img, mask):
    # Original experiment's 20% horizontal flip; disabled by default.
    # Directional class semantics should be checked before augmentation studies.
    flip = tf.random.uniform([]) > 0.8
    img = tf.cond(flip, lambda: tf.image.flip_left_right(img), lambda: img)
    mask = tf.cond(flip, lambda: tf.image.flip_left_right(mask), lambda: mask)
    return img, mask

def create_dataset(split, shuffle=False, augment_data=False):
    image_files, mask_files = paired_paths(dataset_path / "images" / split, dataset_path / "annotations" / split)
    ds = tf.data.Dataset.from_tensor_slices((image_files, mask_files))
    if shuffle:
        ds = ds.shuffle(len(image_files), seed=SEED)
    ds = ds.map(process_image, num_parallel_calls=tf.data.AUTOTUNE)
    if augment_data:
        ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_dataset = create_dataset("train", shuffle=True, augment_data=USE_AUGMENTATION)
val_dataset = create_dataset("val")
test_dataset = create_dataset("test")

In [ ]:
def decode_mask(mask):
    """
    Converts mask (Height ,Width , 1) to RGB color mask
    """
    mask = np.squeeze(mask)
    height, width = mask.shape
    color_mask = np.zeros((height, width, 3), dtype=np.uint8)
    for class_id, color in enumerate(COLORS):
        color_mask[mask == class_id] = color
    return color_mask

def visualize_sample(dataset):
    """
    Visualizing a random sample of dataset
    """
    # Take 1 batch from dataset
    for images, masks in dataset.take(1):
        idx = random.randint(0, images.shape[0]-1)
        img = images[idx].numpy()
        mask = masks[idx].numpy()
        color_mask = decode_mask(mask)

        plt.figure(figsize=(12, 4))
        plt.subplot(1, 3, 1)
        plt.imshow(img)
        plt.title("Image")
        plt.axis("off")

        plt.subplot(1, 3, 2)
        plt.imshow(np.squeeze(mask), cmap="gray")
        plt.title("Mask (raw)")
        plt.axis("off")

        plt.subplot(1, 3, 3)
        plt.imshow(color_mask)
        plt.title("Mask (colorized)")
        plt.axis("off")

        plt.show()
        break

In [ ]:
visualize_sample(train_dataset)

## U-Net architecture

Four encoder levels (64, 128, 256, 512 filters), a 1024-filter bottleneck, skip connections, transposed convolutions and a nine-class softmax output.

In [ ]:
def conv_block(inputs, filters):
    """
    A basic building block:
    - Two convolutional layers
    - Each followed by Batch Normalization and ReLU activation (applying non-linearity)
    """
    x = layers.Conv2D(filters, (3,3), padding="same")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Conv2D(filters, (3,3), padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    return x


def encoder_block(inputs, filters):
    """
    Encoder block (Downsampling path):
    - Apply conv_block
    - Apply MaxPooling
    - Returns:
      x : feature map before pooling (for skip connection later)
      p : pooled feature map (to continue encoder path)
    """
    x = conv_block(inputs, filters)
    p = layers.MaxPooling2D((2,2))(x)
    return x, p


def decoder_block(inputs, skip_features, filters):
    """
    Decoder block (Upsampling path):
    - Apply transpose convolution
    - Concatenate with corresponding skip connection from encoder
    - Apply conv_block
    """
    x = layers.Conv2DTranspose(filters, (2,2), strides=2, padding="same")(inputs)
    x = layers.Concatenate()([x, skip_features])  # Skip connection
    x = conv_block(x, filters)
    return x

In [ ]:
def build_unet(input_shape=(IMG_HEIGHT, IMG_WIDTH, 3), num_classes=NUM_CLASSES):
    """
    Building the full U-Net architecture.

    Args :
    input_shape : Shape of input images (H, W, C)
    num_classes : Number of segmentation classes

    Returns:
    model
    """
    inputs = layers.Input(shape=input_shape)

    ####################################################
    ############## Encoder (Downsampling) ##############
    ####################################################

    s1, p1 = encoder_block(inputs, 64)
    s2, p2 = encoder_block(p1, 128)
    s3, p3 = encoder_block(p2, 256)
    s4, p4 = encoder_block(p3, 512)


    ####################################################
    ############ Bridge (Bottom of the "U") ############
    ####################################################

    b1 = conv_block(p4, 1024)


    ####################################################
    #### Decoder (Upsampling with skip connections) ####
    ####################################################

    d1 = decoder_block(b1, s4, 512) # Connect to s4
    d2 = decoder_block(d1, s3, 256) # Connect to s3
    d3 = decoder_block(d2, s2, 128) # Connect to s2
    d4 = decoder_block(d3, s1, 64)  # Connect to s1


    ####################################################
    ################### Output layer ###################
    ####################################################

    outputs = layers.Conv2D(num_classes, (1,1), padding="same", activation="softmax")(d4)   # Class probabilities per pixel


    # Build model
    model = Model(inputs, outputs, name="U-Net")
    return model


model = build_unet()
model.summary()

## Training

Adam (learning rate 1e-4), sparse categorical cross-entropy, pixel accuracy and confusion-matrix mIoU. The default is a new 18-epoch run, not an exact reconstruction of the original interactive session. Lower `BATCH_SIZE` if memory is limited.

In [ ]:
@tf.keras.utils.register_keras_serializable(package="FacialSegmentation")
class MeanIoU(tf.keras.metrics.Metric):
    def __init__(self, num_classes=NUM_CLASSES, name="mIoU", **kwargs):
        super().__init__(name=name, **kwargs)
        self.num_classes = num_classes

        # This gets updated after each batch
        self.cm = self.add_weight(
            name="conf_matrix",
            shape=(num_classes, num_classes),
            initializer="zeros",
            dtype=tf.float32
        )

    def update_state(self, y_true, y_pred, sample_weight=None):
        """
        Update the confusion matrix after each batch

        Args :
        y_true : Ground truth mask tensor of shape (B, H, W, 1).
                 Contains integer class labels.
        y_pred : Model predictions of shape (B, H, W, C).
                 Contains class probabilities (softmax output).
        """

        # Remove last dimension from y_true and cast to int
        y_true = tf.cast(tf.squeeze(y_true, axis=-1), tf.int32)

        # Convert probabilities to predicted class indices
        y_pred = tf.argmax(y_pred, axis=-1, output_type=tf.int32)

        # Flatten both true and predicted masks into 1D vectors
        y_true = tf.reshape(y_true, [-1])
        y_pred = tf.reshape(y_pred, [-1])

        # Compute confusion matrix for this batch
        cm = tf.math.confusion_matrix(
            y_true, y_pred, num_classes=self.num_classes, dtype=tf.float32
        )

        # Accumulate confusion matrix into the class variable
        self.cm.assign_add(cm)

    def result(self):
        """
        Compute final Mean Intersection over Union from the accumulated confusion matrix.
        IoU_k = TP / (TP + FP + FN) for each class
        mIoU  = average across all classes
        """
        cm = self.cm

        # True positives are diagonal elements
        tp = tf.linalg.diag_part(cm)

        # False positives: sum of column - TP
        fp = tf.reduce_sum(cm, axis=0) - tp

        # False negatives: sum of row - TP
        fn = tf.reduce_sum(cm, axis=1) - tp

        # Denominator: TP + FP + FN for each class
        denom = tp + fp + fn

        # Per-class IoU
        iou_per_class = tf.math.divide_no_nan(tp, denom)

        # Return mean IoU across all classes
        return tf.reduce_mean(iou_per_class)

    def reset_state(self):
        # Reset the confusion matrix (at the start of each epoch)
        self.cm.assign(tf.zeros_like(self.cm))
    def get_config(self):
        return {**super().get_config(), "num_classes": self.num_classes}

In [ ]:
miou_metric = MeanIoU(num_classes=NUM_CLASSES, name="mIoU")

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=["accuracy", miou_metric]
)

In [ ]:
# Existing checkpoints are protected from accidental overwriting.
model_path = MODEL_DIR / "unet.keras"
if model_path.exists():
    raise FileExistsError(f"Choose a new checkpoint filename: {model_path}")
history = model.fit(train_dataset, validation_data=val_dataset, epochs=EPOCHS)
model.save(model_path)
print("Saved model:", model_path)

## Validation and test evaluation

In [ ]:
CLASS_NAMES = [
    "Background", "Person", "Skin", "Left Brow", "Right Brow",
    "Left Eye", "Right Eye", "Lips", "Teeth"
]

def evaluate_iou(model, dataset, num_classes=9, class_names=CLASS_NAMES):
    cm = tf.zeros((num_classes, num_classes), dtype=tf.float32)

    for images, masks in dataset:
        # Predict
        preds = model.predict(images, verbose=0)
        y_pred = tf.argmax(preds, axis=-1)
        y_true = tf.squeeze(masks, axis=-1)

        # Flatten
        y_pred = tf.reshape(tf.cast(y_pred, tf.int32), [-1])
        y_true = tf.reshape(tf.cast(y_true, tf.int32), [-1])

        # Update confusion matrix
        batch_cm = tf.math.confusion_matrix(
            y_true, y_pred, num_classes=num_classes, dtype=tf.float32
        )
        cm += batch_cm

    # IoU = TP / (TP+FP+FN)
    tp = tf.linalg.diag_part(cm)
    fp = tf.reduce_sum(cm, axis=0) - tp
    fn = tf.reduce_sum(cm, axis=1) - tp
    denom = tp + fp + fn
    iou_per_class = tf.math.divide_no_nan(tp, denom)


    miou = float(tf.reduce_mean(iou_per_class).numpy())
    per_class = {class_names[i]: float(iou_per_class[i].numpy()) for i in range(num_classes)}

    return miou, per_class

In [ ]:
import json
evaluation = {}
for split, ds in (("validation", val_dataset), ("test", test_dataset)):
    miou, per_class = evaluate_iou(model, ds)
    evaluation[split] = {"mean_iou": miou, "per_class_iou": per_class}
    print(f"{split} mIoU: {miou:.4f}")
    for name, value in per_class.items():
        print(f"  {name}: {value:.4f}")
(OUTPUT_DIR / "evaluation.json").write_text(json.dumps(evaluation, indent=2), encoding="utf-8")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, key, title in zip(axes, ("loss", "accuracy"), ("Loss", "Pixel accuracy")):
    ax.plot(history.history[key], label="Training")
    ax.plot(history.history["val_" + key], label="Validation")
    ax.set(xlabel="Epoch index", ylabel=title, title=title)
    ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "learning-curves.png", dpi=150)
plt.show()

## Predictions and transparent overlays

This section consolidates the separate overlay notebook from the original project.

In [ ]:
for images, masks in test_dataset.take(1):
    predictions = np.argmax(model.predict(images, verbose=0), axis=-1)
    count = min(4, images.shape[0])
    fig, axes = plt.subplots(count, 4, figsize=(12, 3 * count), squeeze=False)
    for i in range(count):
        image_rgb = (images[i].numpy() * 255).astype(np.uint8)
        mask_rgb = decode_mask(predictions[i])
        overlay_rgb = cv2.addWeighted(image_rgb, 0.6, mask_rgb, 0.4, 0)
        panels = (image_rgb, decode_mask(masks[i].numpy()), mask_rgb, overlay_rgb)
        for ax, panel, title in zip(axes[i], panels, ("Input", "Ground truth", "Prediction", "Overlay")):
            ax.imshow(panel)
            ax.set_title(title)
            ax.axis("off")
        cv2.imwrite(str(OUTPUT_DIR / f"overlay-{i}.png"), cv2.cvtColor(overlay_rgb, cv2.COLOR_RGB2BGR))
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "segmentation-examples.png", dpi=150)
    plt.show()